### Imports

In [1]:
import os
import json
import tempfile
import subprocess
from itertools import islice

import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download


/Users/aneeshkalla/anaconda3/envs/driving/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Template Code

In [2]:
#NOTE: YOU WILL NEED TO MAKE SURE YOU HAVE THE FILE "credentials.json" IN ORDER TO ACCESS THE API
import io
from googleapiclient.http import MediaIoBaseDownload



from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle
import os
import io # Required for downloading file content
import mimetypes # Required for guessing MIME type during upload
#NOTE: You might have to do pip install on a few google libraries before you can use this code
# Define the permissions scope
# 'drive.file' scope grants access only to files created or opened by this app.
SCOPES = ['https://www.googleapis.com/auth/drive']

def get_drive_service():
    """Authenticates the user and returns the Google Drive API service object."""
    creds = None
    # 1. Load existing credentials (token.pickle)
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)
            
    # 2. If no valid creds, refresh or perform OAuth login
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # Refresh expired token silently
            creds.refresh(Request())
        else:
            # Perform full login flow, requires 'credentials.json' in the same directory
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
            
        # 3. Save the new/refreshed token
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)

    service = build('drive', 'v3', credentials=creds)
    return service

def upload_to_drive(file_path, folder_id=None):
    """
    Uploads a file to Google Drive.
    
    :param file_path: The path to the local file to upload.
    :param folder_id: The Google Drive ID of the target folder. None for root.
    :return: The ID of the uploaded file, or None on failure.
    """
    try:
        service = get_drive_service()
        
        # 1. Guess the MIME type from the file extension
        mime_type, _ = mimetypes.guess_type(file_path)
        if not mime_type:
            mime_type = 'application/octet-stream' # Default if type cannot be guessed

        # 2. Set file metadata (name and parent folder)
        file_metadata = {'name': os.path.basename(file_path)}
        if folder_id:
            file_metadata['parents'] = [folder_id]
            
        # 3. Create media object for upload
        media = MediaFileUpload(file_path, mimetype=mime_type, resumable=True)
        
        # 4. Execute the upload
        uploaded = service.files().create(
            body=file_metadata, 
            media_body=media, 
            fields='id'
        ).execute()
        
        file_id = uploaded.get('id')
        print(f"✅ Successfully uploaded '{os.path.basename(file_path)}'. ID: {file_id}")
        return file_id
        
    except FileNotFoundError:
        print(f"❌ Error: The file was not found at path: {file_path}")
        return None
    except Exception as e:
        print(f"❌ An error occurred during upload: {e}")
        return None

def download_from_drive(file_id, local_file_path):
    """
    Downloads a file from Google Drive using its file ID.
    TO FIND THE FILE ID ON GOOGLE DRIVE, SIMPLY GO TO THE GOOGLE DRIVE WEBSITE, CLICK ON THE THING YOU WANT, AND THE LONG STRING OF NUMBERS AND CHARACTERS AFTER "/folders" IN THE WEB ADDRESS IS IT
    :param file_id: The ID of the file to download from Drive.
    :param local_file_path: The path and name to save the file locally.
    :return: True on success, False on failure.
    """
    try:
        service = get_drive_service()

        # 1. Execute the download request using alt='media'
        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)

        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Downloading... {int(status.progress() * 100)}%")

        # ✅ Extract raw bytes from the BytesIO buffer
        file_content = fh.getvalue()

        if isinstance(file_content, dict):
            print("⚠️ Drive returned a dictionary instead of bytes — likely an error:")
            print(file_content)   # <-- this shows the error message from Drive
            return False          # Stop here before writing bad data

        # 2. Write the downloaded byte content to a local file
        with open(local_file_path, 'wb') as f:
            f.write(file_content)

        print(f"⬇️ Successfully downloaded file ID {file_id} to '{local_file_path}'.")
        return True

    except Exception as e:
        print(f"❌ An error occurred during download for file ID {file_id}: {e}", flush=True)
        return False




In [3]:
import os
import cv2
from huggingface_hub import HfFileSystem

# === CONFIG ===
TEMP_DIR = "temp_videos"
os.makedirs(TEMP_DIR, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN")  # set via environment variable
fs = HfFileSystem(token=HF_TOKEN)

def load_video(row):
    """
    Given a DataFrame row (with 'source' and 'filepath'),
    downloads the video if needed and returns the local file path.

    - if source == 'huggingface': downloads via fsspec (fs) into TEMP_DIR
    - if source == 'googledrive': downloads using other_info['fileID'] and the existing
      Google Drive helper functions (download_from_drive)
    """
    source = row["source"]
    filepath = row["filepath"]

    # Always keep the same filename
    filename = os.path.basename(filepath)
    local_path = os.path.join(TEMP_DIR, filename)

    if source == "huggingface":
        # Download if not already present
        if not os.path.exists(local_path):
            print(f"Downloading {filepath} from Hugging Face...")
            with fs.open(filepath, "rb") as src, open(local_path, "wb") as dst:
                dst.write(src.read())
        return os.path.abspath(local_path)

    elif source == "googledrive":
        # Expect 'other_info' to contain the file ID
        try:
            other_info = row["other_info"]
            file_id = other_info["fileID"]
        except Exception:
            raise ValueError("For 'googledrive' source, row['other_info']['fileID'] is required.")

        # Download only if not already saved locally
        if not os.path.exists(local_path):
            print(f"Downloading {filename} from Google Drive (fileID={file_id})...")
            success = download_from_drive(file_id, local_path)
            if not success:
                raise RuntimeError(f"Failed to download file ID {file_id} from Google Drive.")

        return os.path.abspath(local_path)

    else:
        raise NotImplementedError(f"Source '{source}' not supported yet.")

def script1(video_path):
    """
    Simple video player using OpenCV.
    Plays until the end or until 'q' is pressed.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return

    print(f"Playing video: {video_path}")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow("Video", frame)
        if cv2.waitKey(25) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

def delete_video(video_path):
    """
    Deletes a local video file (e.g., one downloaded by load_video()).
    Only allows deletion inside the temp_videos folder for safety.
    """
    abs_path = os.path.abspath(video_path)
    temp_dir_abs = os.path.abspath(TEMP_DIR)

    # Ensure safety: only delete inside temp_videos
    if abs_path.startswith(temp_dir_abs) and os.path.exists(abs_path):
        os.remove(abs_path)
        print(f"Deleted: {abs_path}")
    else:
        print(f"Skipped deletion (unsafe path): {abs_path}")


# === Example usage ===
# import pandas as pd
# df = pd.read_hdf("nexar_train_prepared.h5", key="data")
# row = df.iloc[0]
# local_path = load_video(row)
# script1(local_path)
# delete_video(local_path)


### Example Usage

In [5]:
#loading up a saved dataframe
import pandas as pd
df = pd.read_hdf("nexar_train_prepared.h5", key="data")
df

,filepath,source,event_time,event_description,split,other_info
0,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.500,collision,train,"{""time_of_alert"": 18.633, ""light_conditions"": ..."
1,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.800,collision,train,"{""time_of_alert"": 19.233, ""light_conditions"": ..."
2,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,20.101,collision,train,"{""time_of_alert"": 20.068, ""light_conditions"": ..."
3,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.267,collision,train,"{""time_of_alert"": 18.333, ""light_conditions"": ..."
4,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,21.146,collision,train,"{""time_of_alert"": 18.196, ""light_conditions"": ..."
...,...,...,...,...,...,...
1495,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1496,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1497,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1498,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."


In [6]:
row = df.iloc[0]
local_path = load_video(row)
script1(local_path) #random function that displays the video clip
delete_video(local_path)

Playing video: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4
Deleted: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4


In [ ]:
import pandas as pd
import concurrent.futures

def script2(video_path):
    """
    Return the length of a video in seconds as a float.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    if fps <= 0 or frame_count <= 0:
        return None
    return float(frame_count / fps)


# === Batch processing ===
input_path = "nexar_train_prepared.h5"
output_path = "output.h5"
df = pd.read_hdf(input_path, key="data")
results = []
n = 10  # number of clips per batch
for i in range(3):  # run 3 batches
    start = i * n
    end = start + n
    if start >= len(df):
        break
    batch = df.iloc[start:end]
    # Load videos concurrently
    with concurrent.futures.ThreadPoolExecutor(max_workers=n) as executor:
        futures = {executor.submit(load_video, row): idx for idx, row in batch.iterrows()}
        local_paths = {}
        for future in concurrent.futures.as_completed(futures):
            idx = futures[future]
            try:
                local_paths[idx] = future.result()
            except Exception as e:
                print(f"Error loading row {idx}: {e}")
    for idx, local_path in local_paths.items():
        clip_length = script2(local_path)
        row_data = batch.loc[idx].to_dict()
        row_data["output"] = clip_length
        results.append(row_data)
        delete_video(local_path)

output_df = pd.DataFrame(results)
output_df.to_hdf(output_path, key="data", mode="w")

print(f"✅ Output saved to {output_path}")
print(output_df.head())



Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00408.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00128.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00072.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00471.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00205.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00822.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00208.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00457.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00900.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00948.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00641.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00841.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00938.mp4
Deleted: /Users/aneeshkalla/Desktop/A

In [ ]:
# Example: one-row DataFrame for testing
data = [{
    "filepath": "00822.mp4",  # name of the file in Drive
    "source": "googledrive",        # triggers the Drive branch in load_video()
    "event_time": None,
    "event_description": "test_download",
    "split": "demo",
    "other_info": {"fileID": "18ZrK_DsUHZLjJ7qPDDHECaVmcp-fthK8"}  # your file ID
}]
df = pd.DataFrame(data)
# Preview to confirm
print(df)
# ✅ Use your load_video() function
row = df.iloc[0]
local_path = load_video(row)
print("\nDownloaded local file path:", local_path)

    filepath       source event_time event_description split  \
0  00822.mp4  googledrive       None     test_download  demo   

                                        other_info  
0  {'fileID': '18ZrK_DsUHZLjJ7qPDDHECaVmcp-fthK8'}  

Downloaded local file path: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4


In [ ]:

uploaded_id = upload_to_drive("/Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4", folder_id="1dtgKrSeaAyv4WrbGVr8zeoKH_7B_7bPg")


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=221407270655-90npb4apkqrkpfmpehtn4hpfb3971kvn.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A50266%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=TcPl5Z1UcuktxBBo7j1n8CDaBO7F5v&access_type=offline
✅ Successfully uploaded '00822.mp4'. ID: 1CKFu7T83RZNehEPb-IfxpgZkfo-i1TN6


### DINO basic setup

In [4]:
from pathlib import Path


def collision_video_frames(row, frame_interval_seconds=0.5, impact_window_seconds=2.0):
    """Load a clip, sample frames, store them, and return their metadata as a DataFrame."""
    if frame_interval_seconds <= 0:
        raise ValueError("frame_interval_seconds must be > 0")
    if impact_window_seconds < 0:
        raise ValueError("impact_window_seconds must be >= 0")

    video_path = load_video(row)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video at {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        cap.release()
        delete_video(video_path)
        raise RuntimeError("Video FPS could not be determined")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_step = max(1, int(round(frame_interval_seconds * fps)))
    impact_time = row.get("event_time") if hasattr(row, "get") else row["event_time"]
    impact_time = float(impact_time) if impact_time is not None else None
    impact_window_end = None if impact_time is None else impact_time + float(impact_window_seconds)

    images_dir = Path("frames") / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    video_name = Path(video_path).stem
    rows = []

    try:
        for frame_idx in range(0, total_frames, frame_step):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()
            if not success:
                break

            timestamp = frame_idx / fps

            if impact_time is not None and impact_window_end is not None and timestamp > impact_window_end:
                break  # stop once we're past the post-impact window

            label = 0
            if impact_time is not None:
                label = int(impact_time <= timestamp <= impact_window_end)

            filename = f"{video_name}_{round(timestamp, 3)}s.jpg"
            frame_path = images_dir / filename
            cv2.imwrite(str(frame_path), frame)

            rows.append(
                {
                    "video_name": video_name,
                    "frame_path": str(frame_path),
                    "frame_index": frame_idx,
                    "time_seconds": round(timestamp, 3),
                    "label": label,
                }
            )
    finally:
        cap.release()
        delete_video(video_path)

    return pd.DataFrame(rows)


In [ ]:
'''
# Demo run for the first Nexar collision clip
if "df" not in globals():
    df = pd.read_hdf("nexar_train_prepared.h5", key="data")

sample_row = df.iloc[0]
frames_df = collision_video_frames(
    sample_row,
    frame_interval_seconds=0.2,
    impact_window_seconds=2.0,
)
frames_df
'''

Deleted: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4


,video_name,frame_path,frame_index,time_seconds,label
0,00822,frames/images/00822_0.0s.jpg,0,0.0,0
1,00822,frames/images/00822_0.2s.jpg,6,0.2,0
2,00822,frames/images/00822_0.4s.jpg,12,0.4,0
3,00822,frames/images/00822_0.6s.jpg,18,0.6,0
4,00822,frames/images/00822_0.8s.jpg,24,0.8,0
...,...,...,...,...,...
103,00822,frames/images/00822_20.6s.jpg,618,20.6,1
104,00822,frames/images/00822_20.8s.jpg,624,20.8,1
105,00822,frames/images/00822_21.0s.jpg,630,21.0,1
106,00822,frames/images/00822_21.2s.jpg,636,21.2,1


In [5]:
import os
import urllib
from PIL import Image

import torch
import torchvision.transforms.functional as TF
import numpy as np

print("PyTorch version:", torch.__version__)

# --- device selection for Mac ---
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple Silicon GPU (MPS).")
else:
    DEVICE = "cpu"
    print("Using CPU.")

DEVICE


PyTorch version: 2.9.1
Using Apple Silicon GPU (MPS).


'mps'

In [6]:
DINOV3_REPO = "facebookresearch/dinov3"
#MODEL_NAME = "dinov3_vith16plus"  # HP model
MODEL_NAME = "dinov3_vits16plus"


# Optional: allow user to override with a local clone
DINOV3_LOCATION = os.getenv("DINOV3_LOCATION", DINOV3_REPO)
SOURCE = "local" if DINOV3_LOCATION != DINOV3_REPO else "github"

print("Loading model:", MODEL_NAME)
print("From:", DINOV3_LOCATION, "| source:", SOURCE)

model = torch.hub.load(
    repo_or_dir=DINOV3_LOCATION,
    model=MODEL_NAME,
    source=SOURCE,
)

model = model.to(DEVICE).eval()
print("Model loaded on:", DEVICE)


Loading model: dinov3_vits16plus
From: facebookresearch/dinov3 | source: github


Using cache found in /Users/aneeshkalla/.cache/torch/hub/facebookresearch_dinov3_main


Model loaded on: mps


In [7]:
PATCH_SIZE = 16
IMAGE_SIZE = 768

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def load_image_from_url(url: str) -> Image.Image:
    with urllib.request.urlopen(url) as f:
        return Image.open(f).convert("RGB")

def resize_transform(
    image: Image.Image,
    image_size: int = IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
):
    w, h = image.size
    h_patches = image_size // patch_size
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(
        TF.resize(image, (h_patches * patch_size, w_patches * patch_size))
    )

def normalize_image(t: torch.Tensor) -> torch.Tensor:
    return TF.normalize(t, mean=IMAGENET_MEAN, std=IMAGENET_STD)


In [8]:
# Number of layers for DINOv3 variants
MODEL_TO_NUM_LAYERS = {
    "dinov3_vits16": 12,
    "dinov3_vits16plus": 12,
    "dinov3_vitb16": 12,
    "dinov3_vitl16": 24,
    "dinov3_vith16plus": 32,
    "dinov3_vit7b16": 40,
}

N_LAYERS = MODEL_TO_NUM_LAYERS[MODEL_NAME]

def dino_forward_tokens(
    model,
    img_batch: torch.Tensor,
    layers=None,
    device=DEVICE,
):
    """
    Runs DINOv3 and returns a dict for a batch input:
      { layer_index: {"cls": (B, D), "patch": (B, N_patches, D)} }
    """

    if layers is None:
        layers = [N_LAYERS - 1]  # last layer only

    img_batch = img_batch.to(device)

    with torch.no_grad():  # CPU / MPS friendly
        feats = model.get_intermediate_layers(
            img_batch,
            n=layers,
            reshape=False,
            norm=True,
        )

    outputs = {}

    for layer_idx, tok in zip(layers, feats):
        tok = tok.detach().cpu()  # (B, num_tokens, dim)
        cls_tok = tok[:, 0, :]
        patch_tok = tok[:, 1:, :]
        outputs[layer_idx] = {
            "cls": cls_tok,
            "patch": patch_tok,
        }

    return outputs


In [22]:
# Prepare frames for DINO processing
batch_size = 1
all_frame_paths = frames_df['frame_path'].tolist()
images = []
for path in all_frame_paths:
    img = Image.open(path).convert('RGB')
    img_t = normalize_image(resize_transform(img))
    images.append(img_t)


In [23]:
import tqdm

cls_tokens_list = []
patch_tokens_list = []

for start_idx in tqdm.tqdm(range(0, len(images), batch_size)):
    batch_imgs = torch.stack(images[start_idx:start_idx + batch_size], dim=0)
    print(f'Processing batch from {start_idx} to {start_idx + batch_imgs.shape[0] - 1}')

    batch_tokens = dino_forward_tokens(model, batch_imgs, layers=[N_LAYERS - 1])
    layer_idx = list(batch_tokens.keys())[0]
    cls_tokens_list.append(batch_tokens[layer_idx]['cls'])
    patch_tokens_list.append(batch_tokens[layer_idx]['patch'])

cls_tokens = torch.cat(cls_tokens_list, dim=0)
patch_tokens = torch.cat(patch_tokens_list, dim=0)
print(f'CLS tokens shape: {cls_tokens.shape}')
print(f'Patch tokens shape: {patch_tokens.shape}')

# Attach tokens back to frames_df
frames_df['cls_token'] = list(cls_tokens.numpy())
frames_df['patch_tokens'] = list(patch_tokens.numpy())
frames_df.head()


  0%|          | 0/41 [00:00<?, ?it/s]

Processing batch from 0 to 0


  2%|▏         | 1/41 [00:00<00:35,  1.14it/s]

Processing batch from 1 to 1


  5%|▍         | 2/41 [00:01<00:31,  1.22it/s]

...

Processing batch from 12 to 12


 32%|███▏      | 13/41 [00:10<00:22,  1.24it/s]

Processing batch from 13 to 13


 32%|███▏      | 13/41 [00:11<00:24,  1.15it/s]


KeyboardInterrupt: 

# Training Dino

In [4]:
#helper functions

from pathlib import Path
import tqdm
import os
import urllib
from PIL import Image

import torch
import torchvision.transforms.functional as TF
import numpy as np
import os
import urllib
from PIL import Image

import torch
import torchvision.transforms.functional as TF
import numpy as np

print("PyTorch version:", torch.__version__)

# --- device selection for Mac ---
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple Silicon GPU (MPS).")
else:
    DEVICE = "cpu"
    print("Using CPU.")

DEVICE

def collision_video_frames(row, frame_interval_seconds=0.5, impact_window_seconds=2.0):
    if frame_interval_seconds <= 0:
        raise ValueError("frame_interval_seconds must be > 0")
    if impact_window_seconds < 0:
        raise ValueError("impact_window_seconds must be >= 0")

    video_path = load_video(row)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video at {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        cap.release()
        delete_video(video_path)
        raise RuntimeError("Video FPS could not be determined")

    frame_step = max(1, int(round(frame_interval_seconds * fps)))
    impact_time = float(row["event_time"]) if row["event_time"] is not None else None
    impact_window_end = None if impact_time is None else impact_time + float(impact_window_seconds)

    images_dir = Path("frames") / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    video_name = Path(video_path).stem
    rows = []

    try:
        frame_idx = 0
        while True:
            success, frame = cap.read()
            if not success:
                break

            if frame_idx % frame_step != 0:
                frame_idx += 1
                continue

            timestamp = frame_idx / fps

            if impact_time is not None and impact_window_end is not None and timestamp > impact_window_end:
                break

            label = 0
            if impact_time is not None:
                label = int(impact_time <= timestamp <= impact_window_end)

            filename = f"{video_name}_{round(timestamp, 3)}s.jpg"
            frame_path = images_dir / filename
            cv2.imwrite(str(frame_path), frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

            rows.append(
                {
                    "video_name": video_name,
                    "frame_path": str(frame_path),
                    "frame_index": frame_idx,
                    "time_seconds": round(timestamp, 3),
                    "label": label,
                }
            )

            frame_idx += 1
    finally:
        cap.release()
        delete_video(video_path)

    return pd.DataFrame(rows)

'''
example usage

if "df" not in globals():
    df = pd.read_hdf("nexar_train_prepared.h5", key="data")

sample_row = df.iloc[0]
frames_df = collision_video_frames(
    sample_row,
    frame_interval_seconds=0.2,
    impact_window_seconds=2.0,
)
frames_df


'''

PATCH_SIZE = 16
IMAGE_SIZE = 768

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def load_image_from_url(url: str) -> Image.Image:
    with urllib.request.urlopen(url) as f:
        return Image.open(f).convert("RGB")

def resize_transform(
    image: Image.Image,
    image_size: int = IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
):
    w, h = image.size
    h_patches = image_size // patch_size
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(
        TF.resize(image, (h_patches * patch_size, w_patches * patch_size))
    )

def normalize_image(t: torch.Tensor) -> torch.Tensor:
    return TF.normalize(t, mean=IMAGENET_MEAN, std=IMAGENET_STD)




DINOV3_REPO = "facebookresearch/dinov3"
#MODEL_NAME = "dinov3_vith16plus"  # HP model
MODEL_NAME = "dinov3_vits16plus"


# Optional: allow user to override with a local clone
DINOV3_LOCATION = os.getenv("DINOV3_LOCATION", DINOV3_REPO)
SOURCE = "local" if DINOV3_LOCATION != DINOV3_REPO else "github"

print("Loading model:", MODEL_NAME)
print("From:", DINOV3_LOCATION, "| source:", SOURCE)

model = torch.hub.load(
    repo_or_dir=DINOV3_LOCATION,
    model=MODEL_NAME,
    source=SOURCE,
)

model = model.to(DEVICE).eval()
print("Model loaded on:", DEVICE)

MODEL_TO_NUM_LAYERS = {
    "dinov3_vits16": 12,
    "dinov3_vits16plus": 12,
    "dinov3_vitb16": 12,
    "dinov3_vitl16": 24,
    "dinov3_vith16plus": 32,
    "dinov3_vit7b16": 40,
}

N_LAYERS = MODEL_TO_NUM_LAYERS[MODEL_NAME]

'''
example dino usage

if layers is None:
        layers = [N_LAYERS - 1]  # last layer only

    img_batch = img_batch.to(device)

    with torch.no_grad():  # CPU / MPS friendly
        feats = model.get_intermediate_layers(
            img_batch,
            n=layers,
            reshape=False,
            norm=True,
        )

    outputs = {}

    for layer_idx, tok in zip(layers, feats):
        tok = tok.detach().cpu()  # (B, num_tokens, dim)
        cls_tok = tok[:, 0, :]
        patch_tok = tok[:, 1:, :]
        outputs[layer_idx] = {
            "cls": cls_tok,
            "patch": patch_tok,
        }

'''



PyTorch version: 2.9.1
Using Apple Silicon GPU (MPS).
Loading model: dinov3_vits16plus
From: facebookresearch/dinov3 | source: github


Using cache found in /Users/aneeshkalla/.cache/torch/hub/facebookresearch_dinov3_main


Model loaded on: mps


'\nexample dino usage\n\nif layers is None:\n        layers = [N_LAYERS - 1]  # last layer only\n\n    img_batch = img_batch.to(device)\n\n    with torch.no_grad():  # CPU / MPS friendly\n        feats = model.get_intermediate_layers(\n            img_batch,\n            n=layers,\n            reshape=False,\n            norm=True,\n        )\n\n    outputs = {}\n\n    for layer_idx, tok in zip(layers, feats):\n        tok = tok.detach().cpu()  # (B, num_tokens, dim)\n        cls_tok = tok[:, 0, :]\n        patch_tok = tok[:, 1:, :]\n        outputs[layer_idx] = {\n            "cls": cls_tok,\n            "patch": patch_tok,\n        }\n\n'

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import tqdm

# Load the prepared collision DataFrame if not already present
if "df" not in globals():
    df = pd.read_hdf("nexar_train_prepared.h5", key="data")

train_clips_df, test_clips_df = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True
)
print(f"Train clips: {len(train_clips_df)} | Test clips: {len(test_clips_df)}")

Train clips: 1200 | Test clips: 300


In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def build_frames_df(
    clips_df,
    frame_interval_seconds=0.5,
    impact_window_seconds=2.0,
    max_workers=8,
):
    def process_clip(clip_row):
        return collision_video_frames(
            clip_row,
            frame_interval_seconds=frame_interval_seconds,
            impact_window_seconds=impact_window_seconds,
        )

    clip_rows = list(clips_df.iterrows())
    if not clip_rows:
        return pd.DataFrame(columns=[
            "video_name",
            "frame_path",
            "frame_index",
            "time_seconds",
            "label",
        ])

    frame_parts = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_clip, row): idx for idx, (_, row) in enumerate(clip_rows)}
        for future in tqdm.tqdm(as_completed(futures), total=len(futures)):
            frame_parts.append(future.result())

    return pd.concat(frame_parts, ignore_index=True)
'''
train_frames_df = build_frames_df(train_clips_df)
test_frames_df = build_frames_df(test_clips_df)

print("Train frames:", len(train_frames_df), "| label counts:", train_frames_df["label"].value_counts())
train_frames_df.to_hdf("frames.h5", key="train", mode="w")

print("Test frames:", len(test_frames_df), "| label counts:", test_frames_df["label"].value_counts())
test_frames_df.to_hdf("frames.h5", key="test", mode="a")


# Persist extracted frame metadata for reuse
print("Saved frame DataFrames to frames.h5 (keys: train, test)")
'''

'\ntrain_frames_df = build_frames_df(train_clips_df)\ntest_frames_df = build_frames_df(test_clips_df)\n\nprint("Train frames:", len(train_frames_df), "| label counts:", train_frames_df["label"].value_counts())\ntrain_frames_df.to_hdf("frames.h5", key="train", mode="w")\n\nprint("Test frames:", len(test_frames_df), "| label counts:", test_frames_df["label"].value_counts())\ntest_frames_df.to_hdf("frames.h5", key="test", mode="a")\n\n\n# Persist extracted frame metadata for reuse\nprint("Saved frame DataFrames to frames.h5 (keys: train, test)")\n'

In [7]:
frames_store = "frames.h5"  # adjust path if needed
train_frames_df = pd.read_hdf(frames_store, key="train")
test_frames_df = pd.read_hdf(frames_store, key="test")
print(len(train_frames_df), "train frames loaded")
print(len(test_frames_df), "test frames loaded")

70628 train frames loaded
17715 test frames loaded


In [8]:
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image


def frame_to_tensor(img: Image.Image) -> torch.Tensor:
    # Resize to multiples of patch size and normalize with ImageNet stats
    tensor_img = resize_transform(img, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE)
    return normalize_image(tensor_img)


class FrameDataset(Dataset):
    def __init__(self, frames_df: pd.DataFrame, transform=frame_to_tensor):
        self.df = frames_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["frame_path"]).convert("RGB")
        tensor_img = self.transform(img)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return tensor_img, label


class BalancedBatchSampler(Sampler):
    def __init__(self, zero_indices, one_indices, num_class0, num_class1, batches_per_epoch, seed=42):
        self.zero_indices = list(zero_indices)
        self.one_indices = list(one_indices)
        self.num_class0 = num_class0
        self.num_class1 = num_class1
        self.batches_per_epoch = batches_per_epoch
        self.seed = seed
        self.epoch = 0

    def __iter__(self):
        import numpy as np

        rng = np.random.default_rng(self.seed + self.epoch)
        for _ in range(self.batches_per_epoch):
            batch = []
            batch += rng.choice(
                self.zero_indices,
                size=self.num_class0,
                replace=len(self.zero_indices) < self.num_class0,
            ).tolist()
            batch += rng.choice(
                self.one_indices,
                size=self.num_class1,
                replace=len(self.one_indices) < self.num_class1,
            ).tolist()
            rng.shuffle(batch)
            yield batch
        self.epoch += 1

    def __len__(self):
        return self.batches_per_epoch


def make_balanced_train_loader(
    frames_df: pd.DataFrame,
    num_class0_per_batch: int,
    num_class1_per_batch: int,
    batches_per_epoch: int = 12,
    num_workers: int = 2,
):
    dataset = FrameDataset(frames_df)
    zero_indices = [i for i, lbl in enumerate(dataset.df["label"]) if lbl == 0]
    one_indices = [i for i, lbl in enumerate(dataset.df["label"]) if lbl == 1]
    if not zero_indices or not one_indices:
        raise ValueError("Both classes must be present to build a balanced loader.")

    batch_sampler = BalancedBatchSampler(
        zero_indices,
        one_indices,
        num_class0=num_class0_per_batch,
        num_class1=num_class1_per_batch,
        batches_per_epoch=batches_per_epoch,
    )

    return DataLoader(
        dataset,
        batch_sampler=batch_sampler,
        num_workers=num_workers,
        pin_memory=DEVICE != "cpu",
    )

from torch.utils.data import Subset

def make_eval_loader(
    frames_df: pd.DataFrame,
    batch_size: int = 1,
    num_workers: int = 2,
):
    dataset = FrameDataset(frames_df.reset_index(drop=True))
    eval_subset = Subset(dataset, list(range(min(1000, len(dataset)))))

    return DataLoader(
        eval_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=DEVICE != "cpu",
    )

In [9]:
import torch.nn as nn

# Infer feature dimension from the loaded DINO backbone
FEATURE_DIM = getattr(model, "num_features", getattr(model, "embed_dim", None))
if FEATURE_DIM is None:
    raise ValueError("Could not infer feature dimension from the DINO model.")

import torch
import torch.nn as nn

class AttnPool1D(nn.Module):
    """
    Learned attention pooling over a sequence of tokens.
    Uses a single learned query token to attend over patch tokens.
    """
    def __init__(self, dim: int, num_heads: int = 4):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim))   # (1, 1, D)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            batch_first=True,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, N, D) patch tokens
        returns: (B, D) pooled feature
        """
        B, N, D = x.shape
        q = self.query.expand(B, -1, -1)    # (B, 1, D)
        pooled, _ = self.attn(q, x, x)      # (B, 1, D)
        return pooled.squeeze(1)            # (B, D)


class DinoDualPathBinaryClassifier(nn.Module):
    """
    - Frozen DINOv3 backbone
    - Path 1: deeper MLP on CLS token (global path)
    - Path 2: attention pooling over patch tokens + MLP (local path)
    - Final logit = cls_logit + patch_logit
    """
    def __init__(
        self,
        backbone,
        feature_dim: int,
        n_layers: int,
        layers=None,
        dropout: float = 0,
        patch_heads: int = 4,
    ):
        super().__init__()
        self.backbone = backbone
        self.layers = layers or [n_layers - 1]  # default: last layer

        # freeze DINO backbone
        for p in self.backbone.parameters():
            p.requires_grad = False

        # ---- CLS path: deeper MLP ----
        self.cls_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(feature_dim, feature_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(feature_dim // 2, feature_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(feature_dim // 4, 1),   # scalar logit from CLS path
        )

        # ---- Patch path: attention pooling + MLP ----
        self.patch_pool = AttnPool1D(feature_dim, num_heads=patch_heads)

        self.patch_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(feature_dim, feature_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(feature_dim // 2, feature_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(feature_dim // 4, 1),   # scalar logit from patch path
        )

    def forward(self, x: torch.Tensor):
        """
        x: (B, 3, H, W) preprocessed frames (already resized & normalized)
        """
        feats = self.backbone.get_intermediate_layers(
            x,
            n=self.layers,
            reshape=False,
            norm=True,
        )
        tokens = feats[-1]                 # (B, T, D) where T = 1 + extra + patches
        cls_tok = tokens[:, 0, :]         # (B, D)
        patch_tok = tokens[:, 1:, :]      # (B, T-1, D) (registers+patches, that's fine)

        # CLS path
        cls_logit = self.cls_head(cls_tok).squeeze(-1)        # (B,)

        # Patch path: attention pooling over tokens
        patch_pooled = self.patch_pool(patch_tok)             # (B, D)
        patch_logit = self.patch_head(patch_pooled).squeeze(-1)  # (B,)

        # Combine the two paths at the final node
        logit = cls_logit + patch_logit                       # (B,)

        return logit, cls_logit, patch_logit, cls_tok, patch_tok


def compute_metrics(targets, probs):
    preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(targets, preds) if len(targets) else 0.0
    try:
        auc = roc_auc_score(targets, probs) if len(set(targets)) > 1 else float("nan")
    except ValueError:
        auc = float("nan")
    cm = confusion_matrix(targets, preds, labels=[0, 1]) if len(targets) else np.zeros((2, 2), dtype=int)
    return {"accuracy": acc, "auc": auc, "confusion_matrix": cm}


def run_epoch(loader, model, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_targets, all_probs = [], []

    iterable = tqdm.tqdm(loader, total=len(loader), desc="train" if is_train else "eval")

    for batch in iterable:
        imgs, labels = batch
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.set_grad_enabled(is_train):
            logits, cls_logit, patch_logit, cls_tok, patch_tok = model(imgs)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * len(labels)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_targets.extend(labels.detach().cpu().numpy().tolist())

    avg_loss = running_loss / max(1, len(loader.dataset))
    metrics = compute_metrics(np.array(all_targets), np.array(all_probs))
    metrics["loss"] = avg_loss
    return metrics

In [10]:
train_frames_df#[train_frames_df["label"]==1]

,video_name,frame_path,frame_index,time_seconds,label
0,01970,frames/images/01970_0.0s.jpg,0,0.0,0
1,01970,frames/images/01970_0.5s.jpg,15,0.5,0
2,01970,frames/images/01970_1.0s.jpg,30,1.0,0
3,01970,frames/images/01970_1.5s.jpg,45,1.5,0
4,01970,frames/images/01970_2.0s.jpg,60,2.0,0
...,...,...,...,...,...
70623,01720,frames/images/01720_38.0s.jpg,1140,38.0,0
70624,01720,frames/images/01720_38.5s.jpg,1155,38.5,0
70625,01720,frames/images/01720_39.0s.jpg,1170,39.0,0
70626,01720,frames/images/01720_39.5s.jpg,1185,39.5,0


In [16]:
# Example training/testing loop
train_loader = make_balanced_train_loader(
    train_frames_df,
    num_class0_per_batch=4,
    num_class1_per_batch=4,
    batches_per_epoch=10,
    num_workers=0,
)

test_loader = make_eval_loader(test_frames_df, batch_size=8, num_workers=0)




In [17]:

classifier = DinoDualPathBinaryClassifier(backbone=model, feature_dim=FEATURE_DIM, n_layers=N_LAYERS).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=3e-4)

In [19]:
epochs = 1
for epoch in range(1, epochs + 1):
    train_stats = run_epoch(train_loader, classifier, criterion, optimizer)


    print(f"Epoch {epoch}/{epochs}")
    print("  Train -> loss: {loss:.4f}, acc: {accuracy:.3f}, auc: {auc}, cm:{confusion_matrix}".format(**train_stats))
    if epoch%100 == 0:
        test_stats = run_epoch(test_loader, classifier, criterion, optimizer=None)
        print("  Test  -> loss: {loss:.4f}, acc: {accuracy:.3f}, auc: {auc}, cm:{confusion_matrix}".format(**test_stats))

train:   0%|          | 0/10 [00:00<?, ?it/s]/Users/aneeshkalla/anaconda3/envs/driving/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
train: 100%|██████████| 10/10 [01:10<00:00,  7.06s/it]

Epoch 1/1
  Train -> loss: 0.0005, acc: 0.825, auc: 0.890625, cm:[[36  4]
 [10 30]]


In [ ]:
# Save model weights (and optionally optimizer state)
#v0: mlp with 2 layers on CLS token
#v1: mlp with 5 layers on both CLS and patch tokens, attn pooling on patch tokens
save_path = "dino_classifier_89auc_v1_12.4.3:24.pt"
torch.save(
    {
        "model_state": classifier.state_dict(),
        "optimizer_state": optimizer.state_dict(),
    },
    save_path,
)
print(f"Saved to {save_path}")


Saved to dino_classifier_89auc_v1_12.4.3:24.pt


# model evaluation

In [21]:
checkpoint = torch.load("dino_classifier_89auc_v1_12.4.3:24.pt", map_location=DEVICE)
classifier.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
classifier.to(DEVICE).eval()  # set eval as needed
print("Model and optimizer restored")


Model and optimizer restored


In [22]:
test_clips_df

,filepath,source,event_time,event_description,split,other_info
1116,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1368,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
422,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,20.717,collision,train,"{""time_of_alert"": 18.833, ""light_conditions"": ..."
413,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.122,collision,train,"{""time_of_alert"": 18.162, ""light_conditions"": ..."
451,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,18.933,collision,train,"{""time_of_alert"": 18.033, ""light_conditions"": ..."
...,...,...,...,...,...,...
983,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
799,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1265,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1150,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."


### Collision threshold sweep

In [24]:
import numpy as np


def evaluate_clip_flags(
    row,
    model,
    thresholds,
    frame_interval_seconds=0.5,
    impact_window_seconds=2.0,
    batch_size=1,
):
    video_path = load_video(row)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        delete_video(video_path)
        raise RuntimeError(f"Could not open video at {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        cap.release()
        delete_video(video_path)
        raise RuntimeError("Video FPS could not be determined")

    impact_time = float(row["event_time"]) if row["event_time"] is not None else None
    if impact_time is None:
        cap.release()
        delete_video(video_path)
        return None  # skip clips without impact time

    video_name = row.get("video_name", Path(video_path).stem)
    window_end = impact_time + float(impact_window_seconds)
    window_start = max(0.0, impact_time - float(impact_window_seconds))
    frame_step = max(1, int(round(frame_interval_seconds * fps)))

    frames, timestamps = [], []
    frame_idx = 0
    try:
        while True:
            success, frame = cap.read()
            if not success:
                break

            if frame_idx % frame_step != 0:
                frame_idx += 1
                continue

            timestamp = frame_idx / fps
            if timestamp > window_end:
                break  # stop evaluating beyond +impact_window_seconds

            img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frames.append(frame_to_tensor(img))
            timestamps.append(timestamp)

            frame_idx += 1
    finally:
        cap.release()
        delete_video(video_path)

    if not frames:
        return {"video_name": video_name, "results": []}

    probs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(frames), batch_size):
            batch = torch.stack(frames[i : i + batch_size]).to(DEVICE)
            logits, cls_logit, patch_logit, cls_tok, patch_tok = model(batch)
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())

    clip_results = []
    for ts, prob in zip(timestamps, probs):
        in_window = window_start <= ts <= window_end
        clip_results.append((ts, prob, in_window))
    return {"video_name": video_name, "results": clip_results}


thresholds = np.arange(0.05, 1.0, 0.05)
stats = {
    t: {
        "clips_caught": 0,
        "clips_missed": 0,
        "misfire_frames": 0,
        "misfire_clips": set(),
    }
    for t in thresholds
}

clip_counter = 0
for _, clip_row in tqdm.tqdm(test_clips_df.iterrows(), total=len(test_clips_df), desc="threshold sweep"):
    clip_counter += 1
    clip_out = evaluate_clip_flags(clip_row, classifier, thresholds)
    if not clip_out or not clip_out["results"]:
        continue

    video_name = clip_out["video_name"]
    clip_preds = clip_out["results"]

    caught_clip_flags = {t: False for t in thresholds}
    misfire_clip_flags = {t: False for t in thresholds}
    misfire_frame_counts = {t: 0 for t in thresholds}

    for ts, prob, in_window in clip_preds:
        for t in thresholds:
            pred = prob >= t
            if pred:
                if in_window:
                    caught_clip_flags[t] = True
                else:
                    misfire_frame_counts[t] += 1
                    misfire_clip_flags[t] = True

    for t in thresholds:
        if caught_clip_flags[t]:
            stats[t]["clips_caught"] += 1
        else:
            stats[t]["clips_missed"] += 1

        stats[t]["misfire_frames"] += misfire_frame_counts[t]
        if misfire_clip_flags[t]:
            stats[t]["misfire_clips"].add(video_name)

    if clip_counter % 1 == 0:
        print(f"Processed {clip_counter} clips...")
        temp_results = []
        for t in thresholds:
            entry = stats[t].copy()
            entry["threshold"] = round(float(t), 2)
            entry["misfire_clips"] = len(entry["misfire_clips"])
            temp_results.append(entry)
        temp_df = pd.DataFrame(temp_results).sort_values("threshold")
        display(temp_df)

results = []
for t in thresholds:
    entry = stats[t].copy()
    entry["threshold"] = round(float(t), 2)
    entry["misfire_clips"] = len(entry["misfire_clips"])
    results.append(entry)

threshold_results_df = pd.DataFrame(results).sort_values("threshold")
threshold_results_df

threshold sweep:   0%|          | 0/300 [00:00<?, ?it/s]

Deleted: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/01910.mp4
Processed 1 clips...


,clips_caught,clips_missed,misfire_frames,misfire_clips,threshold
0,0,1,80,1,0.05
1,0,1,80,1,0.10
2,0,1,30,1,0.15
3,0,1,12,1,0.20
4,0,1,6,1,0.25
5,0,1,5,1,0.30
6,0,1,5,1,0.35
7,0,1,5,1,0.40
8,0,1,4,1,0.45
9,0,1,4,1,0.50


threshold sweep:   0%|          | 1/300 [01:06<5:33:38, 66.95s/it]

Deleted: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/01401.mp4
Processed 2 clips...


...

,clips_caught,clips_missed,misfire_frames,misfire_clips,threshold
0,10,14,1447,24,0.05
1,10,14,1447,24,0.10
2,10,14,1180,24,0.15
3,10,14,863,24,0.20
4,10,14,668,24,0.25
5,9,15,559,23,0.30
6,9,15,477,21,0.35
7,9,15,411,20,0.40
8,9,15,344,19,0.45
9,9,15,282,19,0.50


threshold sweep:   8%|▊         | 24/300 [21:06<4:35:03, 59.80s/it]

Deleted: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/02090.mp4


threshold sweep:   8%|▊         | 24/300 [21:35<4:08:16, 53.97s/it]


KeyboardInterrupt: 